In [1]:
import pandas as pd

In [9]:
# Create the Movie Dataset
movies = pd.DataFrame({
    "movie_id": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "title": [
        "Galaxy Mission", "Love in Nairobi", "Final Battle", "Family Weekend",
        "Future World", "Broken Hearts", "City Detectives", "Space Warriors",
        "Wedding Weekend", "Hidden Truth"
    ],
    "Action":  [1, 0, 1, 0, 1, 0, 1, 1, 0, 0],
    "Comedy":  [0, 0, 0, 1, 0, 0, 0, 0, 1, 0],
    "Drama":   [0, 1, 0, 0, 0, 1, 1, 0, 0, 1],
    "Romance": [0, 1, 0, 0, 0, 1, 0, 0, 1, 0],
    "SciFi":   [1, 0, 0, 0, 1, 0, 0, 1, 0, 0]
})

# Inspect Dataset
print("Dataset Head:")
display(movies.head())

print("Dataset Shape:", movies.shape)
print("\nDataset Info:")
movies.info()

from google.colab import files

movies.to_csv("movies.csv", index=False)
files.download("movies.csv")

Dataset Head:


,movie_id,title,Action,Comedy,Drama,Romance,SciFi
0,1,Galaxy Mission,1,0,0,0,1
1,2,Love in Nairobi,0,0,1,1,0
2,3,Final Battle,1,0,0,0,0
3,4,Family Weekend,0,1,0,0,0
4,5,Future World,1,0,0,0,1


Dataset Shape: (10, 7)

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   movie_id  10 non-null     int64 
 1   title     10 non-null     object
 2   Action    10 non-null     int64 
 3   Comedy    10 non-null     int64 
 4   Drama     10 non-null     int64 
 5   Romance   10 non-null     int64 
 6   SciFi     10 non-null     int64 
dtypes: int64(6), object(1)
memory usage: 692.0+ bytes


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  How many movies are in the dataset? There are 10 movies.
  
   
  How many genre features exist? There are 5 genre features (Action, Comedy, Drama, Romance, SciFi).   

  What does one row represent? One row represents a single movie along with its unique ID, title, and binary genre feature profile.

In [3]:
features = ["Action", "Comedy", "Drama", "Romance", "SciFi"]
movie_features = movies[features]
display(movie_features)

# Inspect "Galaxy Mission" profile
galaxy_mission = movies[movies["title"] == "Galaxy Mission"]
display(galaxy_mission)

,Action,Comedy,Drama,Romance,SciFi
0,1,0,0,0,1
1,0,0,1,1,0
2,1,0,0,0,0
3,0,1,0,0,0
4,1,0,0,0,1
5,0,0,1,1,0
6,1,0,1,0,0
7,1,0,0,0,1
8,0,1,0,1,0
9,0,0,1,0,0


,movie_id,title,Action,Comedy,Drama,Romance,SciFi
0,1,Galaxy Mission,1,0,0,0,1


The 1s indicate that Galaxy Mission belongs to the Action and SciFi genres. The 0s indicate the absence of Comedy, Drama, and Romance.

In [4]:
ratings = pd.DataFrame({
    "movie_id": [1, 2, 3, 5],
    "rating": [5, 2, 4, 5]
})

user_movies = ratings.merge(movies, on="movie_id")
display(user_movies[["title", "rating", "Action", "Comedy", "Drama", "Romance", "SciFi"]])

,title,rating,Action,Comedy,Drama,Romance,SciFi
0,Galaxy Mission,5,1,0,0,0,1
1,Love in Nairobi,2,0,0,1,1,0
2,Final Battle,4,1,0,0,0,0
3,Future World,5,1,0,0,0,1


The user strongly prefers Action and SciFi movies (rated 4 and 5 stars), while showing a low preference for Drama and Romance (rated 2 stars for Love in Nairobi)

In [5]:
rated_features = user_movies[features]
user_ratings = user_movies["rating"]

# Weight features by ratings
weighted_features = rated_features.mul(user_ratings, axis=0)
display(weighted_features)

# Sum up weighted features to construct the user profile
user_profile = weighted_features.sum()
print("Raw User Profile:")
print(user_profile.sort_values(ascending=False))

# Normalize the user profile
user_profile_normalized = user_profile / user_profile.sum()
print("\nNormalized User Profile:")
print(user_profile_normalized)

,Action,Comedy,Drama,Romance,SciFi
0,5,0,0,0,5
1,0,0,2,2,0
2,4,0,0,0,0
3,5,0,0,0,5


Raw User Profile:
Action     14
SciFi      10
Drama       2
Romance     2
Comedy      0
dtype: int64

Normalized User Profile:
Action     0.500000
Comedy     0.000000
Drama      0.071429
Romance    0.071429
SciFi      0.357143
dtype: float64


Which two genres appear strongest for this user?

Action and SciFi are the strongest. These genres appear in the user's highest-rated movies (Galaxy Mission, Final Battle, Future World)

In [6]:
# Score every movie
movies["recommendation_score"] = movie_features.mul(user_profile_normalized).sum(axis=1)

# Rank movies
ranked_movies = movies.sort_values("recommendation_score", ascending=False)

# Remove movies already watched
watched_movies = ratings["movie_id"]
recommendations = ranked_movies[~ranked_movies["movie_id"].isin(watched_movies)]

# Top 3 Recommendations
top_3 = recommendations[["title", "recommendation_score"]].head(3)
display(top_3)

,title,recommendation_score
7,Space Warriors,0.857143
6,City Detectives,0.571429
5,Broken Hearts,0.142857


 **pace Warriors:** Recommended because it possesses both Action and SciFi features, matching the user's top two genre preferences.  


 **City Detectives:** Recommended because it has an Action feature (the user's strongest preference score).  


  **Hidden Truth:** Recommended because it has a Drama feature, which carries a small non-zero weight in the profile

In [7]:
def recommend_movies(movies_df, ratings_df, feature_cols, n=3):
    user_movies = ratings_df.merge(movies_df, on="movie_id")
    rated_features = user_movies[feature_cols]
    weighted_features = rated_features.mul(user_movies["rating"], axis=0)

    user_profile = weighted_features.sum()
    user_profile = user_profile / user_profile.sum()

    scores = movies_df[feature_cols].mul(user_profile).sum(axis=1)

    results = movies_df.copy()
    results["recommendation_score"] = scores

    watched = ratings_df["movie_id"]
    unwatched = results[~results["movie_id"].isin(watched)]

    return unwatched.sort_values("recommendation_score", ascending=False).head(n)

# Test User B
user_b_ratings = pd.DataFrame({
    "movie_id": [2, 4, 6, 9],
    "rating": [5, 4, 5, 4]
})

print("User B Recommendations:")
display(recommend_movies(movies, user_b_ratings, features))

User B Recommendations:


,movie_id,title,Action,Comedy,Drama,Romance,SciFi,recommendation_score
9,10,Hidden Truth,0,0,1,0,0,0.3125
6,7,City Detectives,1,0,1,0,0,0.3125
2,3,Final Battle,1,0,0,0,0,0.0000


Comparison: User A and User B received completely different recommendations. User B rated Romance, Drama, and Comedy highly, leading to a profile dominated by these genres rather than Action and SciFi

**What will the model learn?** It will learn that the user only likes Action and SciFi.   


**Which genres will dominate?** Action (50%) and SciFi (50%).  


**What could go wrong?**
Over-specialization . The system will never recommend drama, comedy, or romance movies even if the user might enjoy them.


**Is one rating enough?** No, a single rating produces a narrow, biased profile that leads to poor recommendation diversity
  



In [8]:
# Add unique movie
new_movie = {
    "movie_id": 11,
    "title": "Nairobi Heist",
    "Action": 1,
    "Comedy": 1,
    "Drama": 0,
    "Romance": 0,
    "SciFi": 0
}

movies = pd.concat([movies, pd.DataFrame([new_movie])], ignore_index=True)

# Custom User Test
my_user = pd.DataFrame({
    "movie_id": [1, 4, 7, 11],
    "rating": [5, 5, 4, 5]
})

print("Custom User Top 5 Recommendations:")
display(recommend_movies(movies, my_user, features, n=5))

Custom User Top 5 Recommendations:


,movie_id,title,Action,Comedy,Drama,Romance,SciFi,recommendation_score
4,5,Future World,1,0,0,0,1,0.575758
7,8,Space Warriors,1,0,0,0,1,0.575758
2,3,Final Battle,1,0,0,0,0,0.424242
8,9,Wedding Weekend,0,1,0,1,0,0.303030
1,2,Love in Nairobi,0,0,1,1,0,0.121212


***Limitations of Content-Based Recommenders***


**Feature Sparsity & Oversimplification:** Relying on only five binary genre tags misses essential metadata like actors, directors, mood, or release year.  


**Over-specialization:** Users are only recommended items similar to what they have already liked, limiting discovery.  


**Cold Start Problem:** New users with few or no ratings cannot get personalized, accurate recommendations.

**If two users have watched exactly the same movies but gave them different ratings, should this recommender produce the
same recommendations for both users? Use your Week 3 model to explain your answer.**


No, the recommender will not produce the same recommendations.

Although both users watched the exact same set of movies, feature weighting depends directly on the ratings assigned to each item. Higher ratings scale up the presence of features in those movies, while lower ratings reduce their impact. Consequently, different rating patterns produce distinct normalized user profiles, resulting in different overall scores and movie rankings